# 🧠 3D Medical Image Segmentation with MONAI & PyTorch
### Spleen Segmentation on the Medical Segmentation Decathlon (Task09_Spleen)

---

**Hardware Target:** Google Colab · NVIDIA T4 GPU (16 GB VRAM)  
**Core Frameworks:** [PyTorch](https://pytorch.org/) · [MONAI](https://monai.io/)  

---

## 🗺️ Notebook Overview

| # | Section | Key Concepts |
|---|---------|--------------|
| 1 | Environment Setup | pip, seeds, device detection |
| 2 | Dataset & Preprocessing | MSD, 3D transforms, patch sampling |
| 3 | Data Loaders | CacheDataset, ThreadDataLoader |
| 4 | Model Architecture | 3D U-Net (MONAI) |
| 5 | Loss / Optimizer / Metrics | DiceCELoss, AdamW, CosineAnnealingLR |
| 6 | Training Loop | AMP, GradScaler, best-model checkpointing |
| 7 | Inference & Evaluation | Sliding Window Inference, post-processing |
| 8 | Visualization | Axial / Coronal / Sagittal comparisons |

---

### 🔬 Clinical Context
The **spleen** is a compact, homogeneous abdominal organ making it an excellent benchmark task for 3D segmentation pipelines.  
Accurate spleen delineation is critical for surgical planning, radiation therapy, and trauma assessment.  
This notebook trains a fully volumetric deep-learning model that ingests raw CT volumes and outputs voxel-wise binary masks.


## ⚙️ Section 1 – Environment Setup

We install **MONAI** (with nibabel for NIfTI I/O), verify the GPU, and lock all random seeds to ensure **fully reproducible** results.

> **Why MONAI?**  
> MONAI is a PyTorch-based framework purpose-built for medical imaging. It provides:
> - Ready-made 3D transforms that respect spatial metadata (voxel spacing, orientation).
> - Memory-efficient patch-based training utilities.
> - Production-grade model zoo (U-Net, SegResNet, SwinUNETR, …).


In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
# Run once per Colab session; subsequent runs can comment this out.
!pip install -q "monai[nibabel,tqdm]" matplotlib


In [ ]:
# ── Standard library imports ────────────────────────────────────────────────
import os
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore")

# ── Third-party ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

# ── MONAI core ───────────────────────────────────────────────────────────────
import monai
from monai.apps import DecathlonDataset
from monai.config import print_config
from monai.data import (
    CacheDataset,
    DataLoader,
    ThreadDataLoader,
    decollate_batch,
)
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    # Loading & channel
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    # Spatial
    Spacingd,
    Orientationd,
    RandFlipd,
    RandRotate90d,
    RandShiftIntensityd,
    # Intensity
    ScaleIntensityRanged,
    CropForegroundd,
    # Patch sampling
    RandCropByPosNegLabeld,
    # Post-processing
    AsDiscreted,
    Compose,
    Invertd,
    KeepLargestConnectedComponentd,
)
from monai.utils import first, set_determinism

print_config()


In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED: int = 42

def seed_everything(seed: int = SEED) -> None:
    """Lock all sources of randomness for deterministic training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_determinism(seed=seed)
    # Deterministic CUDA ops (slight perf cost, worth it for research)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ── Device detection ─────────────────────────────────────────────────────────
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅  GPU detected : {gpu_name}  ({vram_gb:.1f} GB VRAM)")
    print(f"    CUDA version : {torch.version.cuda}")
else:
    print("⚠️  No GPU found — training will be very slow on CPU.")

print(f"\n🔧  Active device : {DEVICE}")
print(f"🌱  Random seed   : {SEED}")


## 📂 Section 2 – Dataset: MSD Task09_Spleen

### Medical Segmentation Decathlon (MSD)
The MSD is the largest publicly available multi-organ segmentation benchmark.  
**Task09_Spleen** contains:
- **41 portal-venous phase CT** scans (training) + **20 test** scans.
- NIfTI-format images (.nii.gz) with isotropic-ish voxel spacing ~1 mm³.
- Binary labels: `0 = background`, `1 = spleen`.

### Download via MONAI `DecathlonDataset`
`DecathlonDataset` is a convenience class that:
1. Downloads the compressed dataset tarball from the MSD server.
2. Extracts it to a local directory.
3. Parses `dataset.json` to build train / validation file lists automatically.

> **📌 Colab path:** All data lands under `/content/data/` to avoid ephemeral `/tmp/` issues.

---

### 🧩 3D Preprocessing Pipeline

Each CT volume passes through the following deterministic transforms:

| Transform | Purpose |
|-----------|---------|
| `LoadImaged` | Read NIfTI using nibabel; preserve spatial metadata |
| `EnsureChannelFirstd` | Add batch/channel dim: `(H,W,D)` → `(1,H,W,D)` |
| `Spacingd` | Resample to **1.5 × 1.5 × 2.0 mm** isotropic spacing |
| `Orientationd` | Reorient to **RAS** canonical axes |
| `ScaleIntensityRanged` | Clip HU `[-57, 164]` then rescale to `[0, 1]` |
| `CropForegroundd` | Remove large air background to save GPU memory |
| `RandCropByPosNegLabeld` | Sample **96³ patches** centred on foreground/background |
| `RandFlipd` | Stochastic left-right flipping (augmentation) |
| `RandRotate90d` | Random 90° rotations (augmentation) |
| `RandShiftIntensityd` | Additive intensity jitter (augmentation) |

#### Why patch sampling?
A full spleen CT volume (~512×512×100 voxels) at fp32 would require ~100 MB of GPU memory per sample — before accounting for model activations.  
Extracting **96³ patches** reduces peak VRAM to ~2-3 GB per sample, enabling `batch_size=2` on the T4.


In [ ]:
# ── Configuration constants ───────────────────────────────────────────────────
DATA_ROOT:     Path = Path("/content/data")
CHECKPOINT:    Path = Path("/content/best_model.pth")
NUM_WORKERS:   int  = 2      # Colab typically has 2 CPU cores available
CACHE_RATE:    float = 1.0   # Cache 100 % of training data in RAM (~4-6 GB)

# Patch size — must be divisible by 2^num_pool_layers for U-Net
PATCH_SIZE: Sequence[int] = (96, 96, 96)
# Positive/negative patch ratio (pos=1 means ≥1 foreground voxel in patch)
POS_NEG_RATIO: Tuple[int, int] = (1, 1)
NUM_PATCHES_PER_IMAGE: int = 4   # patches extracted per training volume per epoch

BATCH_SIZE:  int   = 2
MAX_EPOCHS:  int   = 50     # Increase to 300+ for publication-quality results
VAL_EVERY:   int   = 5      # Validate every N epochs (saves time on Colab)
LR:          float = 1e-4

# HU window for spleen in portal-venous CT
HU_MIN: int = -57
HU_MAX: int = 164

DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"📁  Data root : {DATA_ROOT}")
print(f"💾  Checkpoint: {CHECKPOINT}")


In [ ]:
# ── Download Task09_Spleen ────────────────────────────────────────────────────
# DecathlonDataset handles download + extraction automatically.
# section="training" downloads labelled volumes; "test" volumes have no masks.

print("⬇️   Downloading Task09_Spleen (≈1.6 GB — first run only) …")

train_ds_raw = DecathlonDataset(
    root_dir=str(DATA_ROOT),
    task="Task09_Spleen",
    section="training",
    download=True,
    seed=SEED,
    val_frac=0.2,          # 80/20 split → ~33 train, ~8 val
    transform=None,        # transforms applied later via CacheDataset
    cache_rate=0.0,        # no caching here; we use a separate CacheDataset
    num_workers=NUM_WORKERS,
)

val_ds_raw = DecathlonDataset(
    root_dir=str(DATA_ROOT),
    task="Task09_Spleen",
    section="validation",
    download=False,        # already downloaded above
    seed=SEED,
    transform=None,
    cache_rate=0.0,
    num_workers=NUM_WORKERS,
)

print(f"\n✅  Training samples   : {len(train_ds_raw)}")
print(f"✅  Validation samples : {len(val_ds_raw)}")

# Preview a raw data dictionary
sample = train_ds_raw[0]
print(f"\n📋  Raw sample keys    : {list(sample.keys())}")
print(f"    image path         : {sample['image']}")
print(f"    label path         : {sample['label']}")


In [ ]:
# ── Transform pipelines ───────────────────────────────────────────────────────

# ——— Training transforms (with data augmentation) ————————————————————————
train_transforms = Compose([
    # 1. I/O
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    EnsureTyped(keys=["image", "label"]),

    # 2. Spatial normalisation
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),   # bilinear for images, NN for labels
    ),
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    # 3. Intensity normalisation (spleen portal-venous HU window)
    ScaleIntensityRanged(
        keys=["image"],
        a_min=HU_MIN, a_max=HU_MAX,
        b_min=0.0,    b_max=1.0,
        clip=True,
    ),

    # 4. Remove large air background to reduce patch sampling waste
    CropForegroundd(keys=["image", "label"], source_key="image"),

    # 5. Random foreground-centred patch extraction (memory-efficient training)
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=POS_NEG_RATIO[0],
        neg=POS_NEG_RATIO[1],
        num_samples=NUM_PATCHES_PER_IMAGE,
        image_key="image",
        image_threshold=0,
    ),

    # 6. Stochastic augmentation
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.10),
    RandRotate90d(keys=["image", "label"], prob=0.10, max_k=3),
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.50),

    EnsureTyped(keys=["image", "label"], device=DEVICE, track_meta=False),
])

# ——— Validation transforms (deterministic — no augmentation) ————————————
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    EnsureTyped(keys=["image", "label"]),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=HU_MIN, a_max=HU_MAX,
        b_min=0.0,    b_max=1.0,
        clip=True,
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    EnsureTyped(keys=["image", "label"], device=DEVICE, track_meta=True),
])

print("✅  Transform pipelines defined.")
print(f"    Train steps : {len(train_transforms.transforms)}")
print(f"    Val   steps : {len(val_transforms.transforms)}")


## 🏗️ Section 3 – CacheDataset & Data Loaders

### Why CacheDataset?
In 3D medical imaging the **I/O bottleneck** dominates training time, not the GPU.  
Reading, decompressing, and resampling a single NIfTI volume can take 5-10 seconds.  
`CacheDataset` performs the *deterministic* part of the pipeline **once** during the first epoch and stores results in RAM, delivering a **5-10× speed-up** for subsequent epochs.

| Dataset | RAM usage | Speedup |
|---------|-----------|---------|
| Plain `Dataset` | ~0 | 1× |
| `CacheDataset (rate=1.0)` | ~4-6 GB | ~8× |

> **⚠️ Colab RAM budget:** The default Colab instance has ~12 GB RAM.  
> With `cache_rate=1.0` for ~33 spleen volumes post-resampling you'll use ~4 GB — safe.  
> Lower `cache_rate` if you hit RAM limits or use a larger dataset.


In [ ]:
# ── CacheDataset — pre-compute deterministic transforms ──────────────────────

# We pass the *raw* DecathlonDataset dictionaries (just file paths) and apply
# transforms inside CacheDataset for maximum caching efficiency.

train_files = [{"image": s["image"], "label": s["label"]} for s in train_ds_raw]
val_files   = [{"image": s["image"], "label": s["label"]} for s in val_ds_raw]

print("⚙️   Building CacheDataset for training  (this may take 5-10 min) …")
train_ds = CacheDataset(
    data=train_files,
    transform=train_transforms,
    cache_rate=CACHE_RATE,
    num_workers=NUM_WORKERS,
    progress=True,
)

print("\n⚙️   Building CacheDataset for validation …")
val_ds = CacheDataset(
    data=val_files,
    transform=val_transforms,
    cache_rate=1.0,          # val set is small — always cache fully
    num_workers=NUM_WORKERS,
    progress=True,
)

print(f"\n✅  train_ds size : {len(train_ds)}")
print(f"✅  val_ds   size : {len(val_ds)}")


In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
# ThreadDataLoader is a MONAI drop-in replacement for torch DataLoader that
# uses threads instead of processes — avoids forking overhead in Colab.

train_loader = ThreadDataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # 0 = main thread; CacheDataset makes this fast enough
    pin_memory=False,
)

val_loader = ThreadDataLoader(
    val_ds,
    batch_size=1,       # Full volumes during validation (SWI handles memory)
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print(f"✅  Train loader : {len(train_loader)} batches/epoch  (batch_size={BATCH_SIZE})")
print(f"✅  Val   loader : {len(val_loader)} volumes")

# ── Sanity check: inspect a training patch ───────────────────────────────────
sample_batch = first(train_loader)
img_shape   = sample_batch["image"].shape
label_shape = sample_batch["label"].shape

print(f"\n📊  Sample batch:")
print(f"    image  : {img_shape}   dtype={sample_batch['image'].dtype}")
print(f"    label  : {label_shape}  dtype={sample_batch['label'].dtype}")
print(f"    unique label values : {torch.unique(sample_batch['label'])}")


## 🤖 Section 4 – Model Architecture: 3D U-Net

### Architecture Overview

The **3D U-Net** is the _de facto_ standard architecture for volumetric medical image segmentation.  
It follows an **encoder–decoder** design with **skip connections**:

```
Input (1, 96, 96, 96)
    │
    ▼
[Encoder]
  Conv3D(1→16) ─────────────────────────────────────────────┐  skip₁
  ↓ stride 2                                                  │
  Conv3D(16→32) ────────────────────────────────────────────┐ │  skip₂
  ↓ stride 2                                                  │ │
  Conv3D(32→64) ──────────────────────────────────────────┐  │ │  skip₃
  ↓ stride 2                                               │  │ │
  Conv3D(64→128) ────────────────────────────────────────┐ │  │ │  skip₄
  ↓ stride 2                                              │ │  │ │
  [Bottleneck] Conv3D(128→256)                            │ │  │ │
  ↑ upsample                                              │ │  │ │
[Decoder]                                                  │ │  │ │
  ConvTranspose3D(256→128) ◄─ concat ─────────────────────┘ │  │ │
  ConvTranspose3D(128→64)  ◄─ concat ────────────────────── ┘  │ │
  ConvTranspose3D(64→32)   ◄─ concat ───────────────────────── ┘ │
  ConvTranspose3D(32→16)   ◄─ concat ──────────────────────────── ┘
    │
    ▼
  Conv3D(16→2)   [logits for background + spleen]
    │
    ▼
Output (2, 96, 96, 96)
```

### T4 Memory Budget
| Component | Approx. VRAM |
|-----------|-------------|
| Model parameters (fp16) | ~30 MB |
| Activations per batch (AMP fp16) | ~4 GB |
| Optimizer states (AdamW fp32) | ~60 MB |
| **Total** | **~5 GB** ✅ |

AMP(Automatic Mixed Precision) halves activation memory vs. full fp32 training.


In [ ]:
# ── 3D U-Net definition ───────────────────────────────────────────────────────

NUM_INPUT_CHANNELS:  int = 1   # Single-modality CT
NUM_OUTPUT_CLASSES:  int = 2   # Background (0) + Spleen (1)

model: nn.Module = UNet(
    spatial_dims=3,
    in_channels=NUM_INPUT_CHANNELS,
    out_channels=NUM_OUTPUT_CLASSES,
    channels=(16, 32, 64, 128, 256),   # Feature maps at each encoder level
    strides=(2, 2, 2, 2),              # Downsampling strides between levels
    num_res_units=2,                   # Residual units per block (improves grad flow)
    norm="BATCH",                      # Batch normalisation (stable for 3D)
    dropout=0.0,                       # Disabled; add ~0.1 if overfitting
).to(DEVICE)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🏗️   Model architecture : MONAI 3D U-Net")
print(f"    Total parameters   : {total_params:,}")
print(f"    Trainable params   : {trainable_params:,}")
print(f"    Input shape        : (B, {NUM_INPUT_CHANNELS}, {PATCH_SIZE[0]}, {PATCH_SIZE[1]}, {PATCH_SIZE[2]})")
print(f"    Output shape       : (B, {NUM_OUTPUT_CLASSES}, {PATCH_SIZE[0]}, {PATCH_SIZE[1]}, {PATCH_SIZE[2]})")

# ── Quick forward pass to verify shapes ───────────────────────────────────────
with torch.no_grad():
    dummy = torch.zeros(1, 1, *PATCH_SIZE, device=DEVICE)
    out   = model(dummy)
    print(f"\n🔍  Forward pass OK : {dummy.shape} → {out.shape}")


## ⚖️ Section 5 – Loss Function, Optimizer & Metrics

### Loss: DiceCELoss

The **DiceCELoss** is a weighted sum of two complementary losses:

$$\mathcal{L} = \lambda_{\text{Dice}} \cdot \mathcal{L}_{\text{Dice}} + \lambda_{\text{CE}} \cdot \mathcal{L}_{\text{CE}}$$

| Component | Strength | Weakness |
|-----------|----------|---------|
| **Dice Loss** | Handles severe class imbalance natively | Noisy gradients early in training |
| **Cross-Entropy** | Stable, pixel-wise gradient signal | Ignores region-overlap quality |
| **Combined** | Best of both worlds ✅ | — |

In spleen CT the foreground (spleen) occupies only ~0.5–2% of all voxels — pure CE would predict "all background" trivially.

### Optimizer: AdamW
AdamW applies **decoupled weight decay** (L2 regularisation), avoiding the bias that standard Adam introduces when combining adaptive LR with weight decay.

### Scheduler: CosineAnnealingLR
The cosine schedule smoothly anneals LR from `η_max` to `η_min`:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{t\pi}{T}\right)$$

This avoids oscillations and typically improves final Dice by 1-2 points over a flat LR.

### Metric: DiceMetric
$$\text{Dice} = \frac{2 |P \cap G|}{|P| + |G|}$$

`include_background=False` excludes class-0 (background) from averaging — otherwise the metric is dominated by trivially correct background predictions.


In [ ]:
# ── Loss function ─────────────────────────────────────────────────────────────
loss_function = DiceCELoss(
    to_onehot_y=True,      # Convert integer label → one-hot internally
    softmax=True,          # Apply softmax to raw logits before loss
    lambda_dice=0.5,       # Equal weighting between Dice and CE
    lambda_ce=0.5,
)

# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-5,
)

# ── LR Scheduler ──────────────────────────────────────────────────────────────
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
    eta_min=LR / 100,      # Anneal down to 1% of initial LR
)

# ── Validation metric ─────────────────────────────────────────────────────────
dice_metric = DiceMetric(
    include_background=False,   # Exclude background class (class 0)
    reduction="mean",           # Average Dice across batch items
    get_not_nans=False,
)

# ── AMP GradScaler ────────────────────────────────────────────────────────────
# GradScaler prevents fp16 underflow by dynamically scaling the loss before
# backpropagation and unscaling gradients before the optimizer step.
scaler = GradScaler(enabled=torch.cuda.is_available())

# ── Post-processing transforms (applied to logits during validation) ──────────
from monai.transforms import AsDiscrete
post_pred   = Compose([AsDiscrete(argmax=True, to_onehot=NUM_OUTPUT_CLASSES)])
post_label  = Compose([AsDiscrete(to_onehot=NUM_OUTPUT_CLASSES)])

print("✅  Loss          : DiceCELoss (λ_Dice=0.5, λ_CE=0.5)")
print(f"✅  Optimizer     : AdamW (lr={LR}, wd=1e-5)")
print(f"✅  Scheduler     : CosineAnnealingLR (T_max={MAX_EPOCHS})")
print( "✅  Metric        : DiceMetric (background excluded)")
print( "✅  AMP GradScaler: enabled")


## 🚀 Section 6 – Training Loop (AMP-Accelerated)

### Automatic Mixed Precision (AMP)

Modern GPUs (T4, A100) have dedicated **Tensor Cores** that execute fp16 matrix operations 2-8× faster than fp32.  
PyTorch's AMP lets you exploit this with minimal code changes:

```python
with autocast():          # Cast eligible ops to fp16 automatically
    output = model(input)
    loss   = criterion(output, target)

scaler.scale(loss).backward()    # Scale loss to prevent fp16 underflow
scaler.step(optimizer)           # Unscale → clip → step
scaler.update()                  # Adjust scale factor dynamically
```

**VRAM savings:** AMP typically halves activation memory, allowing `batch_size=2` on T4 where fp32 would OOM.

### Training Loop Structure

```
For each epoch:
  ├── [Train]
  │     For each batch:
  │       ├── Forward pass (autocast)
  │       ├── Compute DiceCELoss
  │       ├── Backward (GradScaler)
  │       └── Optimizer step + scheduler step
  │
  └── [Validate — every VAL_EVERY epochs]
        For each full volume:
          ├── Sliding Window Inference (autocast)
          ├── Post-process → discrete one-hot
          └── Accumulate DiceMetric
        Aggregate → save best checkpoint if improved
```


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

# History containers for plotting
train_loss_history: List[float] = []
val_dice_history:   List[Tuple[int, float]] = []   # (epoch, mean_dice)

best_val_dice:  float = 0.0
best_epoch:     int   = 0

# ── Sliding window inference parameters (used during validation) ──────────────
SW_ROI_SIZE   = PATCH_SIZE         # Must match training patch size
SW_SW_BATCH   = 4                  # Sub-windows processed simultaneously in SWI
SW_OVERLAP    = 0.5                # 50% overlap → smoother boundary predictions


def run_validation(model: nn.Module, loader: DataLoader) -> float:
    """
    Run full-volume validation using sliding window inference.
    Returns mean Dice score over all validation volumes (background excluded).
    """
    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for val_data in loader:
            val_inputs  = val_data["image"].to(DEVICE)
            val_labels  = val_data["label"].to(DEVICE)

            # ── Sliding Window Inference ─────────────────────────────────────
            # Full CT volumes are too large to forward-pass in one shot.
            # sliding_window_inference tiles the volume into overlapping ROI
            # patches, runs inference on each, and stitches outputs with
            # Gaussian-weighted blending at patch boundaries.
            with autocast(enabled=torch.cuda.is_available()):
                val_outputs = sliding_window_inference(
                    inputs=val_inputs,
                    roi_size=SW_ROI_SIZE,
                    sw_batch_size=SW_SW_BATCH,
                    predictor=model,
                    overlap=SW_OVERLAP,
                    mode="gaussian",        # Gaussian weighting stitches patches smoothly
                )

            # ── Post-processing ──────────────────────────────────────────────
            val_outputs_post = [post_pred(i)  for i in decollate_batch(val_outputs)]
            val_labels_post  = [post_label(i) for i in decollate_batch(val_labels)]

            # Accumulate metric
            dice_metric(y_pred=val_outputs_post, y=val_labels_post)

    # Aggregate across all validation volumes
    mean_dice: float = dice_metric.aggregate().item()
    dice_metric.reset()
    return mean_dice


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════
print(f"🏋️   Starting training for {MAX_EPOCHS} epochs on {DEVICE} …")
print(f"     Validating every {VAL_EVERY} epochs.")
print(f"     Checkpoint → {CHECKPOINT}\n")
print(f"{'Epoch':>6} | {'Train Loss':>11} | {'Val Dice':>9} | {'LR':>10} | {'Best?':>5}")
print("─" * 55)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss: float  = 0.0
    step:       int    = 0

    for batch_data in train_loader:
        step += 1
        inputs  = batch_data["image"].to(DEVICE)   # (B, 1, 96, 96, 96)
        labels  = batch_data["label"].to(DEVICE)   # (B, 1, 96, 96, 96)

        optimizer.zero_grad(set_to_none=True)

        # ── Forward pass under AMP context ──────────────────────────────────
        with autocast(enabled=torch.cuda.is_available()):
            outputs: torch.Tensor = model(inputs)  # (B, 2, 96, 96, 96)
            loss:    torch.Tensor = loss_function(outputs, labels)

        # ── Scaled backward pass ─────────────────────────────────────────────
        scaler.scale(loss).backward()

        # Check if gradients are populated
        has_grad = any(p.grad is not None for p in model.parameters())
        if not has_grad:
            print("WARNING: No gradients computed!")

        # Gradient clipping (unscale first to get real gradient norms)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    # Step the LR scheduler once per epoch
    scheduler.step()

    avg_loss = epoch_loss / step
    train_loss_history.append(avg_loss)
    current_lr = scheduler.get_last_lr()[0]

    # ── Periodic validation ──────────────────────────────────────────────────
    if epoch % VAL_EVERY == 0 or epoch == MAX_EPOCHS:
        mean_dice = run_validation(model, val_loader)
        val_dice_history.append((epoch, mean_dice))

        is_best = mean_dice > best_val_dice
        if is_best:
            best_val_dice = mean_dice
            best_epoch    = epoch
            torch.save(model.state_dict(), str(CHECKPOINT))

        marker = "⭐" if is_best else "  "
        print(f"{epoch:>6} | {avg_loss:>11.4f} | {mean_dice:>9.4f} | {current_lr:>10.2e} | {marker}")
    else:
        print(f"{epoch:>6} | {avg_loss:>11.4f} | {'—':>9} | {current_lr:>10.2e} |")

print("─" * 55)
print(f"\n🏆  Training complete!")
print(f"    Best Val Dice : {best_val_dice:.4f}  (epoch {best_epoch})")
print(f"    Checkpoint    : {CHECKPOINT}")


## 📈 Section 7 – Training Curves

Plotting the training loss and validation Dice side-by-side allows us to:
- Detect **overfitting** (loss drops but Dice plateaus/drops)
- Identify the **optimal early-stopping** epoch
- Verify the **cosine LR** schedule is working as expected


In [ ]:
# ── Training curve plots ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Summary — 3D Spleen Segmentation", fontsize=14, fontweight="bold")

# ─ Left: Training Loss ────────────────────────────────────────────────────────
axes[0].plot(range(1, MAX_EPOCHS + 1), train_loss_history,
             color="#E63946", linewidth=2, label="Train Loss")
axes[0].axvline(x=best_epoch, color="#2A9D8F", linestyle="--", linewidth=1.5,
                label=f"Best epoch ({best_epoch})")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("DiceCELoss", fontsize=12)
axes[0].set_title("Training Loss", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([1, MAX_EPOCHS])

# ─ Right: Validation Dice ─────────────────────────────────────────────────────
if val_dice_history:
    val_epochs, val_dices = zip(*val_dice_history)
    axes[1].plot(val_epochs, val_dices, color="#457B9D", linewidth=2,
                 marker="o", markersize=5, label="Val Dice (spleen)")
    axes[1].scatter([best_epoch], [best_val_dice], color="#E63946",
                    zorder=5, s=100, label=f"Best: {best_val_dice:.4f}")
    axes[1].axhline(y=best_val_dice, color="#E63946", linestyle=":", alpha=0.5)
    axes[1].set_xlabel("Epoch", fontsize=12)
    axes[1].set_ylabel("Mean Dice Score", fontsize=12)
    axes[1].set_title("Validation Dice (background excluded)", fontsize=12)
    axes[1].set_ylim([0, 1])
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([1, MAX_EPOCHS])

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅  Training curves saved to /content/training_curves.png")


## 🔬 Section 8 – Inference & Evaluation

### Why Sliding Window Inference?

During **training** we use 96³ patches that fit in GPU memory.  
During **inference** we want predictions over the **full CT volume** (e.g. 512×512×100 voxels).  
Naively forwarding the entire volume would OOM on the T4.

`sliding_window_inference` solves this by:
1. Tiling the volume into overlapping **96³ windows**.
2. Running the model on each tile independently.
3. **Stitching** the softmax logit maps back using Gaussian-weighted blending — this eliminates harsh boundary artefacts at tile edges.

```
Full volume (512×512×100)
   ┌───────────────────────────────────────────┐
   │  ┌───────┐                                 │
   │  │ tile₁ │──── model ──→ logits₁           │
   │  └───────┘                                 │
   │      ┌───────┐                             │
   │      │ tile₂ │──── model ──→ logits₂       │
   │      └───────┘                             │  Gaussian blend
   │          ...                               │ ──────────────→ full prediction
   └───────────────────────────────────────────┘
```

### Post-processing

After softmax + argmax we apply `AsDiscreted` to binarise the prediction, then optionally `KeepLargestConnectedComponentd` to remove spurious small-region false positives.

### Final Dice Score

The Dice coefficient is reported **per class** (background excluded) over all validation volumes.


In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
if CHECKPOINT.exists():
    model.load_state_dict(torch.load(str(CHECKPOINT), map_location=DEVICE))
    print(f"✅  Loaded best checkpoint from epoch {best_epoch}  ({CHECKPOINT.name})")
else:
    print("⚠️  No checkpoint found — using current model weights.")

# ── Full inference on the entire validation set ───────────────────────────────
model.eval()
dice_metric.reset()
all_dice_scores: List[float] = []

with torch.no_grad():
    for idx, val_data in enumerate(val_loader):
        val_inputs  = val_data["image"].to(DEVICE)
        val_labels  = val_data["label"].to(DEVICE)

        with autocast(enabled=torch.cuda.is_available()):
            val_outputs = sliding_window_inference(
                inputs=val_inputs,
                roi_size=SW_ROI_SIZE,
                sw_batch_size=SW_SW_BATCH,
                predictor=model,
                overlap=SW_OVERLAP,
                mode="gaussian",
            )

        # Post-process
        val_outputs_post = [post_pred(i)  for i in decollate_batch(val_outputs)]
        val_labels_post  = [post_label(i) for i in decollate_batch(val_labels)]

        # Per-volume Dice
        dice_metric(y_pred=val_outputs_post, y=val_labels_post)
        per_vol_dice = dice_metric.aggregate().item()
        all_dice_scores.append(per_vol_dice)
        dice_metric.reset()

        print(f"  Volume {idx+1:02d} | Dice = {per_vol_dice:.4f}")

# ── Aggregate ─────────────────────────────────────────────────────────────────
mean_dice_final = np.mean(all_dice_scores)
std_dice_final  = np.std(all_dice_scores)

print(f"\n{'─'*40}")
print(f"  Final validation results:")
print(f"  Mean Dice : {mean_dice_final:.4f} ± {std_dice_final:.4f}")
print(f"  Min  Dice : {np.min(all_dice_scores):.4f}")
print(f"  Max  Dice : {np.max(all_dice_scores):.4f}")
print(f"{'─'*40}")
print("\n📊  Clinical reference: expert inter-rater Dice for spleen ≈ 0.93-0.97")


## 🎨 Section 9 – 3D Visualisation: Axial · Coronal · Sagittal

Visualising the three **orthogonal slice planes** provides a complete picture of segmentation quality:

| Plane | Axis | Clinical Relevance |
|-------|------|--------------------|
| **Axial** | Z (inf→sup) | Standard radiological reading plane |
| **Coronal** | Y (ant→post) | Best view for left/right organ comparison |
| **Sagittal** | X (R→L) | Best view for anterior-posterior extent |

### Colour conventions
- **Image:** Grayscale CT in HU-normalised range [0, 1]  
- **Ground Truth:** Semi-transparent **green** overlay  
- **Prediction:** Semi-transparent **red** overlay  

A Dice score is printed in the figure title for each volume.


In [ ]:
# ── Visualisation helper ──────────────────────────────────────────────────────
from matplotlib.colors import ListedColormap

def get_slices_at(
    volume: np.ndarray,
    cx: int, cy: int, cz: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract axial, coronal, and sagittal slices from a 3D volume at a specific coordinate.

    Args:
        volume: 3-D numpy array (H, W, D) or (1, H, W, D).

    Returns:
        Tuple of (axial, coronal, sagittal) 2-D slices.
    """
    if volume.ndim == 4:
        volume = volume[0]      # Remove channel dim: (1,H,W,D) → (H,W,D)
    axial    = volume[:, :, cz]     # H×W plane at depth cz
    coronal  = volume[:, cy, :]     # H×D plane at y cy
    sagittal = volume[cx, :, :]     # W×D plane at x cx
    return axial, coronal, sagittal


def overlay_mask(
    ax: plt.Axes,
    image_slice: np.ndarray,
    gt_slice:   np.ndarray,
    pred_slice: np.ndarray,
    title: str,
) -> None:
    """Render CT slice with colour-coded ground-truth and prediction overlays."""
    ax.imshow(image_slice.T, cmap="gray", origin="lower", vmin=0, vmax=1)

    # Create a custom overlay combining gt and pred
    # 0 = background (transparent)
    # 1 = GT only (Green)
    # 2 = Pred only (Red)
    # 3 = Both (Yellow)
    combined = np.zeros_like(gt_slice)
    combined[gt_slice > 0] = 1
    combined[pred_slice > 0] = 2
    combined[(gt_slice > 0) & (pred_slice > 0)] = 3

    cmap_custom = ListedColormap(["none", "#00ff00", "#ff0000", "#ffff00"])

    ax.imshow(np.ma.masked_where(combined.T == 0, combined.T),
              cmap=cmap_custom, alpha=0.5, origin="lower", vmin=0, vmax=3)

    ax.set_title(title, fontsize=9, pad=4)
    ax.axis("off")


def visualise_prediction(
    image:      np.ndarray,   # (1, H, W, D)  float
    gt_label:   np.ndarray,   # (1, H, W, D)  int
    pred_label: np.ndarray,   # (1, H, W, D)  int
    dice_score: float,
    vol_idx:    int,
) -> None:
    """Plot axial / coronal / sagittal slices for one validation volume."""
    # Find the center of the ground truth organ
    gt_mask = gt_label[0] if gt_label.ndim == 4 else gt_label
    coords = np.argwhere(gt_mask > 0)
    if len(coords) > 0:
        cx, cy, cz = coords.mean(axis=0).astype(int)
    else:
        # Fallback to image center if no organ found
        cx, cy, cz = [s // 2 for s in gt_mask.shape]

    img_ax,  img_cor,  img_sag  = get_slices_at(image, cx, cy, cz)
    gt_ax,   gt_cor,   gt_sag   = get_slices_at(gt_label, cx, cy, cz)
    pred_ax, pred_cor, pred_sag = get_slices_at(pred_label, cx, cy, cz)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("#1a1a2e")

    overall_title = (
        f"Volume {vol_idx+1:02d}  |  Dice = {dice_score:.4f}  |  "
        f"Green = GT only, Red = Pred only, Yellow = Overlap"
    )
    fig.suptitle(overall_title, color="white", fontsize=12, fontweight="bold", y=1.01)

    overlay_mask(axes[0], img_ax,  gt_ax,  pred_ax,  f"Axial (Z={cz})")
    overlay_mask(axes[1], img_cor, gt_cor, pred_cor, f"Coronal (Y={cy})")
    overlay_mask(axes[2], img_sag, gt_sag, pred_sag, f"Sagittal (X={cx})")

    # Legend
    gt_patch   = mpatches.Patch(color="#00ff00", alpha=0.5, label="GT Only")
    pred_patch = mpatches.Patch(color="#ff0000", alpha=0.5, label="Prediction Only")
    over_patch = mpatches.Patch(color="#ffff00", alpha=0.5, label="Overlap (TP)")
    fig.legend(handles=[gt_patch, pred_patch, over_patch], loc="lower center",
               ncol=3, fontsize=10, facecolor="#1a1a2e", labelcolor="white",
               framealpha=0.8, bbox_to_anchor=(0.5, -0.05))

    plt.tight_layout()
    save_path = f"/content/segmentation_vol{vol_idx+1:02d}.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"   💾  Saved → {save_path}")


print("✅  Visualisation helpers defined.")

In [ ]:
# ── Run visualisation on best and worst validation volumes ────────────────────

model.eval()
MAX_DISPLAY: int = min(3, len(val_loader))   # Show up to 3 volumes

collected: List[Dict] = []

with torch.no_grad():
    for idx, val_data in enumerate(val_loader):
        val_inputs  = val_data["image"].to(DEVICE)
        val_labels  = val_data["label"].to(DEVICE)

        with autocast(enabled=torch.cuda.is_available()):
            val_outputs = sliding_window_inference(
                inputs=val_inputs,
                roi_size=SW_ROI_SIZE,
                sw_batch_size=SW_SW_BATCH,
                predictor=model,
                overlap=SW_OVERLAP,
                mode="gaussian",
            )

        # Argmax to get hard prediction
        pred_hard = torch.argmax(val_outputs, dim=1, keepdim=True)

        collected.append({
            "image": val_inputs.cpu().numpy()[0],        # (1,H,W,D)
            "label": val_labels.cpu().numpy()[0],        # (1,H,W,D)
            "pred":  pred_hard.cpu().numpy()[0],         # (1,H,W,D)
            "dice":  all_dice_scores[idx],
            "idx":   idx,
        })

        if len(collected) >= MAX_DISPLAY:
            break

# Sort by Dice — show best, median, worst for clinical coverage
collected.sort(key=lambda x: x["dice"], reverse=True)

print(f"🖼️   Rendering {len(collected)} segmentation figures …\n")

for item in collected:
    visualise_prediction(
        image=item["image"],
        gt_label=item["label"],
        pred_label=item["pred"],
        dice_score=item["dice"],
        vol_idx=item["idx"],
    )

print("\n✅  All figures rendered and saved.")


## 🎞️ Section 10 – Axial Slice Stack (Interactive)

Stepping through **all axial slices** from inferior to superior gives the fullest picture of segmentation fidelity — including boundary behaviour, near-vessel regions, and superior/inferior tips where the spleen tapers.

The cell below renders a static **summary strip** of 8 evenly-spaced axial slices for the first displayed volume, providing a concise but comprehensive visual.


In [ ]:
# ── Multi-slice axial strip ───────────────────────────────────────────────────
from matplotlib.colors import ListedColormap

if collected:
    sample_item = collected[0]   # Best-Dice volume
    img_vol  = sample_item["image"][0]     # (H, W, D) float
    gt_vol   = sample_item["label"][0]     # (H, W, D) int
    pred_vol = sample_item["pred"][0]      # (H, W, D) int

    # Find the range of slices containing the ground truth organ
    coords = np.argwhere(gt_vol > 0)
    if len(coords) > 0:
        z_min, z_max = coords[:, 2].min(), coords[:, 2].max()
        # Add a little padding to the range
        z_min = max(0, z_min - 5)
        z_max = min(img_vol.shape[2] - 1, z_max + 5)
    else:
        D = img_vol.shape[2]
        z_min, z_max = int(D * 0.15), int(D * 0.85)

    N_STRIPS = 8
    slice_indices = np.linspace(z_min, z_max, N_STRIPS, dtype=int)

    fig, axes = plt.subplots(3, N_STRIPS, figsize=(20, 7))
    fig.patch.set_facecolor("#0d1117")
    fig.suptitle(
        f"Axial Slice Strip — Volume {sample_item['idx']+1:02d}  "
        f"|  Dice = {sample_item['dice']:.4f}",
        color="white", fontsize=13, fontweight="bold",
    )

    row_labels = ["CT Image", "Ground Truth", "Prediction"]

    # We will use customized colormaps to ensure bright colors clearly show
    cmap_gt = ListedColormap(["none", "#00ff00"])
    cmap_pred = ListedColormap(["none", "#ff0000"])

    for col, sl in enumerate(slice_indices):
        img_slice = img_vol[:, :, sl]
        gt_slice = gt_vol[:, :, sl]
        pred_slice = pred_vol[:, :, sl]

        for row in range(3):
            ax = axes[row, col]
            ax.axis("off")

            if row == 0:
                # Image only
                ax.imshow(img_slice.T, cmap="gray", origin="lower", vmin=0, vmax=1)
            elif row == 1:
                # Ground truth
                ax.imshow(img_slice.T, cmap="gray", origin="lower", vmin=0, vmax=1)
                ax.imshow(np.ma.masked_where(gt_slice.T == 0, gt_slice.T),
                          cmap=cmap_gt, alpha=0.5, origin="lower", vmin=0, vmax=1)
            else:
                # Prediction
                ax.imshow(img_slice.T, cmap="gray", origin="lower", vmin=0, vmax=1)
                ax.imshow(np.ma.masked_where(pred_slice.T == 0, pred_slice.T),
                          cmap=cmap_pred, alpha=0.5, origin="lower", vmin=0, vmax=1)

            if col == 0:
                ax.set_ylabel(row_labels[row], color="white",
                              fontsize=9, labelpad=6)
                ax.yaxis.set_label_coords(-0.15, 0.5)
                # re-enable y axis label by turning axis completely on is tricky, just use text
                ax.text(-0.15, 0.5, row_labels[row], color="white", fontsize=12,
                        transform=ax.transAxes, ha='right', va='center', rotation=90)

        axes[0, col].set_title(f"z={sl}", color="white", fontsize=10)

    plt.subplots_adjust(wspace=0.03, hspace=0.08)
    save_path = "/content/axial_strip.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"✅  Axial strip saved → {save_path}")

## 📋 Section 11 – Summary & Next Steps

### Results Summary

| Metric | Value |
|--------|-------|
| Dataset | MSD Task09_Spleen |
| Model | MONAI 3D U-Net (16→32→64→128→256 ch) |
| Patch size | 96³ voxels |
| Batch size | 2 |
| Mixed precision | AMP fp16 ✅ |
| Best Val Dice | _see training output_ |
| Inference | Sliding Window (50 % overlap, Gaussian blend) |

---

### 🔭 Next Steps & Extensions

**Architecture upgrades:**
- [ ] Replace U-Net with **SegResNet** (`monai.networks.nets.SegResNet`) — often +1-3 Dice points.
- [ ] Try **SwinUNETR** (transformer-based) for state-of-the-art performance.

**Training improvements:**
- [ ] Increase `MAX_EPOCHS` to 300-500 with a warmup phase.
- [ ] Add **Deep Supervision** to intermediate decoder outputs.
- [ ] Use **TTA (Test-Time Augmentation)**: average predictions across flipped volumes.

**Post-processing:**
- [ ] Apply `KeepLargestConnectedComponentd` to remove spurious small FP regions.
- [ ] Morphological hole-filling for smooth final masks.

**Multi-organ extension:**
- [ ] Swap to **Task07_Pancreas** or **Task03_Liver** to test harder cases.
- [ ] Use MONAI's `one_hot` multi-class pipeline for simultaneous multi-organ segmentation.

**Deployment:**
- [ ] Export to **TorchScript** or **ONNX** for production inference.
- [ ] Wrap inference in a **MONAI Deploy** App Package (MAP) for clinical integration.

---

### 📚 References

1. Antonelli et al. (2022). *The Medical Segmentation Decathlon.* Nature Communications.  
2. Çiçek et al. (2016). *3D U-Net: Learning Dense Volumetric Segmentation from Sparse Annotation.* MICCAI.  
3. MONAI Consortium (2022). *MONAI: Medical Open Network for AI.*  
4. Micikevicius et al. (2018). *Mixed Precision Training.* ICLR.  

---
*Notebook generated for educational and research purposes. Always validate medical AI models against local institutional standards before clinical use.*


## 🪞 Section 12 – Reflection Questions

Use these questions to deepen your understanding of the pipeline. They range from conceptual to research-level — work through them after your first successful training run.

---

### 🔵 Conceptual Foundations

**Q1 — Patch Sampling Bias**
> `RandCropByPosNegLabeld` is configured with `pos=1, neg=1`, meaning 50% of patches are guaranteed to contain at least one foreground (spleen) voxel.
>
> - What would happen to training if you set `pos=0` (only background patches)?
> - Conversely, what is the risk of setting `pos=10, neg=1` (almost exclusively foreground)?
> - How does this ratio relate to the concept of *hard negative mining* in detection pipelines?

---

**Q2 — Loss Function Design**
> This notebook uses `DiceCELoss` with equal weighting (`λ_Dice = λ_CE = 0.5`).
>
> - Dice Loss is *region-based* while Cross-Entropy is *pixel-wise*. Why does combining them produce more stable training than either alone, especially in early epochs?
> - If the spleen occupied only 0.1% of voxels instead of ~1%, how would you adjust the CE component to compensate? What MONAI parameter enables this?
> - Sketch the gradient signal that Dice Loss produces when the prediction is entirely zero (all background). Why can this be problematic?

---

**Q3 — Sliding Window Inference vs. Patch-Based Training**
> During training, 96³ patches are extracted randomly. During inference, `sliding_window_inference` tiles the *entire* volume with 50% overlap.
>
> - Why can't we simply forward-pass the entire volume during inference, even though validation batch size is 1?
> - What artefacts can appear at patch *boundaries* during stitching, and how does `mode="gaussian"` mitigate them compared to `mode="constant"`?
> - Increasing overlap from 0.5 to 0.75 improves prediction smoothness but increases inference time. Can you estimate the computational cost multiplier for a 3D volume when overlap doubles?

---

### 🟡 Implementation & Engineering

**Q4 — Automatic Mixed Precision (AMP)**
> AMP uses fp16 for eligible operations and fp32 for numerically sensitive ones (e.g., softmax, batch norm statistics).
>
> - Why does the `GradScaler` multiply the loss by a large scalar before `backward()`, and then divide gradients before `optimizer.step()`? What failure mode does this prevent?
> - Which layers in the 3D U-Net are *not* cast to fp16 by `autocast`, and why?
> - If you removed `GradScaler` but kept `autocast`, what symptom would you observe in the training loss curve within the first few iterations?

---

**Q5 — CacheDataset and Memory Trade-offs**
> `CacheDataset(cache_rate=1.0)` pre-computes and stores all *deterministic* transforms in RAM.
>
> - The random augmentation transforms (`RandFlipd`, `RandCropByPosNegLabeld`, etc.) are **not** cached — they run fresh each epoch. Why is this architecturally correct?
> - If you are working with a dataset 5× larger (e.g., 165 spleen volumes), and RAM is insufficient for `cache_rate=1.0`, what value would you choose and what is the performance trade-off?
> - `SmartCacheDataset` is an alternative that rotates a fixed cache window. When would it outperform `CacheDataset`?

---

**Q6 — Spacing and Orientation Normalisation**
> `Spacingd` resamples all volumes to a fixed voxel spacing of `(1.5, 1.5, 2.0) mm`, and `Orientationd` reorients to `RAS`.
>
> - The original MSD spleen CTs have variable spacing (roughly 0.6–0.9 mm in-plane, 1.5–8 mm slice thickness). Why is isotropic resampling critical before feeding volumes to a convolutional network?
> - `mode="bilinear"` is used for the image but `mode="nearest"` for the label. What would go wrong if you applied bilinear interpolation to integer segmentation masks?
> - Resampling to a coarser spacing (e.g., 3.0 mm isotropic) would reduce memory and speed up training. What clinical information would be lost, and for which organ shapes would this be most harmful?

---

### 🔴 Research & Clinical Depth

**Q7 — Evaluation Metric Validity**
> The notebook reports Dice coefficient as the primary metric, with `include_background=False`.
>
> - A model that correctly segments 95% of spleen voxels but produces a single large false-positive blob the same size as the spleen would still report a Dice of ~0.63. What complementary metric would catch this failure mode?
> - For a clinical application like pre-surgical planning, is Dice the most clinically meaningful metric? Propose an alternative or additional metric and justify it.
> - Inter-rater variability between expert radiologists on spleen segmentation is approximately Dice 0.93–0.97. What does it mean if your model achieves Dice 0.96 on the test set?

---

**Q8 — Generalisation and Domain Shift**
> This model is trained and evaluated on a single institution's CT protocol from the MSD dataset.
>
> - List three sources of *domain shift* that could degrade performance when deploying to a different hospital's CT scanner.
> - The model was trained with HU window `[-57, 164]`. If a new site uses a different contrast agent protocol that shifts the spleen's HU range to `[30, 220]`, what would you observe in the prediction, and how would you fix it without retraining?
> - How does `Orientationd(axcodes="RAS")` protect against one specific class of domain shift? What would happen without it?

---

**Q9 — Architecture Ablation**
> The U-Net uses 5 encoder levels with channels `(16, 32, 64, 128, 256)` and `num_res_units=2`.
>
> - Removing the skip connections would convert the U-Net into a plain encoder-decoder. Describe the qualitative difference you would expect in the segmentation output, especially near organ boundaries.
> - `num_res_units=2` adds residual connections within each block. How does this help gradient flow in a deep 3D network, and what well-known problem does it address?
> - If you doubled all channel counts to `(32, 64, 128, 256, 512)`, estimate the VRAM increase and predict whether training would still fit on the T4 at `batch_size=2, patch_size=96³`.

---

**Q10 — Open-Ended Design Challenge**
> Imagine you are asked to extend this pipeline to segment **five abdominal organs simultaneously** (liver, spleen, kidneys ×2, pancreas) from the same CT scans.
>
> 1. What changes are needed in the label encoding, model output layer, and loss function?
> 2. The pancreas is highly irregular and occupies ~0.1% of voxels. What sampling strategy modification would you make to ensure it is adequately represented during training?
> 3. Sketch a post-processing pipeline that handles the anatomical constraint that the two kidneys must be spatially separated and the spleen must appear only on the left side of the body.
> 4. How would you report a single aggregate performance number for clinical comparison across five organs with very different sizes and difficulties?

---

> 💡 **Tip:** The most illuminating way to answer the implementation questions (Q4–Q6) is to deliberately break the component in question, re-run training for 5 epochs, and observe what changes in the loss curve, VRAM usage, or output quality.
